In [1]:
import ee
import geemap

ee.Initialize(project="oc-flux")

Map = geemap.Map()

In [2]:
roi = ee.Geometry.Point([79.8033,11.1313]).buffer(5000)

Map.centerObject(roi,12)

Map.addLayer(roi,{},"ROI")

In [3]:
collection = (

ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")

.filterBounds(roi)

.filterDate("2023-01-01","2023-12-31")

)

In [4]:
print(collection.size().getInfo())

22


In [5]:
def maskL8(image):

    qa = image.select("QA_PIXEL")

    cloud = qa.bitwiseAnd(1<<3).eq(0)

    shadow = qa.bitwiseAnd(1<<4).eq(0)

    return image.updateMask(cloud.And(shadow))

In [6]:
collection = collection.map(maskL8)

In [7]:
image = collection.median().clip(roi)

In [8]:
Map.addLayer(

image,

{
"bands":["SR_B4","SR_B3","SR_B2"],
"min":8000,
"max":18000

},

"True Colour"

)

In [9]:
ndwi = image.normalizedDifference(

["SR_B3","SR_B5"]

).rename("NDWI")

In [10]:
water = image.updateMask(ndwi.gt(0))

In [11]:
Map.addLayer(

water,

{

"bands":["SR_B4","SR_B3","SR_B2"],

"min":8000,

"max":18000

},

"Water"

)

In [12]:
predictors = water.select(

[
"SR_B2",
"SR_B3",
"SR_B4",
"SR_B5"
]

)

In [13]:
print(predictors.bandNames().getInfo())

['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5']


In [14]:
prediction = predictors.expression(

"""
0.15 * Blue +
0.35 * Green -
0.20 * Red +
0.50 * NIR
""",

{

"Blue":predictors.select("SR_B2"),

"Green":predictors.select("SR_B3"),

"Red":predictors.select("SR_B4"),

"NIR":predictors.select("SR_B5")

}

).rename("POC")

In [15]:
Map.addLayer(

prediction,

{

"min":6000,

"max":12000,

"palette":[

"blue",

"cyan",

"green",

"yellow",

"red"

]

},

"POC Prediction"

)

In [16]:
print(collection.size().getInfo())

print(image.bandNames().getInfo())

print(prediction.bandNames().getInfo())

22
['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT']
['POC']


In [17]:
Map

Map(center=[11.131305129441879, 79.80330014497318], controls=(WidgetControl(options=['position', 'transparent_…